# AverageField Hardware FIFO Sanity

This notebook tests the native `AverageField` module against the real Spectrum digitizer and GPU from this repository build. It supports both native channel layouts:

- `two_complex`: Spectrum physical channels `[0, 1, 2, 3]` -> two complex fields. Use this for `average`, `average_g1`, and `all_correlators`.
- `one_complex`: Spectrum physical channels `[0, 1]` -> one complex field. Use this only with `result_mode="average"`; cross-channel G1/G2 outputs are intentionally unavailable.

Start external output/trigger devices manually before running the measurement chunk cell.

In [ ]:
from pathlib import Path
import ctypes
import importlib.util
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display

PROJECT = Path.cwd()
if not (PROJECT / "CMakePresets.json").exists() and Path(r"C:\Users\Qop\AverageField").exists():
    PROJECT = Path(r"C:\Users\Qop\AverageField")

QO_REPO = Path(r"C:\Users\Qop\QO-measurements")
if QO_REPO.exists() and str(QO_REPO) not in sys.path:
    sys.path.insert(0, str(QO_REPO))

if hasattr(os, "add_dll_directory"):
    cuda_path = Path(os.environ.get("CUDA_PATH", r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v13.0"))
    for dll_dir in (cuda_path / "bin" / "x64", cuda_path / "bin", Path(r"C:\Windows\System32")):
        if dll_dir.exists():
            os.add_dll_directory(str(dll_dir))

from drivers.Spectrum_m4x import SPCM, SPCM_MODE, SPCM_TRIGGER


def load_averagefield(build_dir=PROJECT / "build" / "windows-qom-ninja"):
    build_dir = Path(build_dir)
    candidates = sorted(build_dir.glob("AverageField*.pyd"))
    candidates += sorted((build_dir / "Release").glob("AverageField*.pyd"))
    if not candidates:
        raise FileNotFoundError(f"No AverageField*.pyd found in {build_dir}")

    module_path = candidates[0]
    spec = importlib.util.spec_from_file_location("AverageField", module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    print("Loaded:", module_path)
    return module


AverageField = load_averagefield()

In [ ]:
# Main test configuration.
# Use one_complex only for average/S21-style tests.
CHANNEL_LAYOUT = "two_complex"  # "two_complex" or "one_complex"
RESULT_MODE = "average_g1"      # "average", "average_g1", or "all_correlators"

DIGITIZER = b"/dev/spcm0"
USE_EXTERNAL_CLOCK = True

DUR_SEG_NS = 1000
N_SEG = 1 << 10
AVERAGES = 1 << 13
SECOND_OVERSAMPLING = 1
CH_AMPLITUDE_MV = 200
DIGITIZER_DELAY_NS = 90
UPDATE_BATCHES = 1

if CHANNEL_LAYOUT == "one_complex" and RESULT_MODE != "average":
    raise ValueError("one_complex only supports RESULT_MODE='average'")

DIG_CHANNELS = [0, 1, 2, 3] if CHANNEL_LAYOUT == "two_complex" else [0, 1]
print("channels:", DIG_CHANNELS)
print("result_mode:", RESULT_MODE)
print("channel_layout:", CHANNEL_LAYOUT)

In [ ]:
dig = SPCM(DIGITIZER)
if USE_EXTERNAL_CLOCK:
    dig.setup_external_clock()

dig_params = {
    "channels": DIG_CHANNELS,
    "ch_amplitude": CH_AMPLITUDE_MV,
    "dur_seg": DUR_SEG_NS,
    "n_avg": 0,
    "n_seg": N_SEG,
    "oversampling_factor": 1,
    "pretrigger": 32,
    "digitizer_delay": DIGITIZER_DELAY_NS,
    "mode": SPCM_MODE.MULTIPLE_FIFO,
    "trig_source": SPCM_TRIGGER.EXT0,
}

dig.set_parameters(dig_params)

print("channels:", dig.channels)
print("sample_rate:", dig.get_sample_rate())
print("segment_size:", dig.get_segment_size())
print("n_seg:", dig.n_seg)
print("driver _bufsize:", getattr(dig, "_bufsize", None))
try:
    print("estimated MiB/s at 1000 ns period:", dig.calc_transfer_speed_mib(1000))
except Exception as exc:
    print("transfer estimate unavailable:", exc)

In [ ]:
afw = AverageField.AverageFieldMeasurer(
    ctypes.addressof(dig.h_card.contents),
    int(AVERAGES),
    int(dig_params["n_seg"]),
    int(SECOND_OVERSAMPLING),
    RESULT_MODE,
    CHANNEL_LAYOUT,
)

afw.set_amplitude(int(dig_params["ch_amplitude"]))
afw.set_calibration(0, 1.0, 0.0, 0.0, 0.0)
if CHANNEL_LAYOUT == "two_complex":
    afw.set_calibration(1, 1.0, 0.0, 0.0, 0.0)
afw.set_firwin(-512.0, 512.0)
afw.set_intermediate_frequency(0.0)

print("afw created")
print("result_mode:", afw.get_result_mode())
print("channel_layout:", afw.get_channel_layout())
print("complex_field_count:", afw.get_complex_field_count())
print("physical_channel_count:", afw.get_physical_channel_count())
print("total_length:", afw.get_total_length())
print("trace_length:", afw.get_trace_length())
print("resampled_trace_length:", afw.get_resampled_trace_length())
print("out_size:", afw.get_out_size())
print("notify_size:", afw.get_notify_size())
print("batches_total:", afw.get_batches_total())
print("averages_total:", afw.get_averages_total())

Start the external trigger/output sequence before running the next cell. The cell starts the digitizer FIFO once, processes `UPDATE_BATCHES` FIFO notifications at a time, and keeps accumulating native output state, so the plots show intermediate averages.

In [ ]:
afw.reset_output()
times = []

afw.start_fifo()
try:
    while afw.get_batches_remaining() > 0:
        batches = min(int(UPDATE_BATCHES), int(afw.get_batches_remaining()))
        t0 = time.perf_counter()
        afw.measure_batches(batches)
        times.append(time.perf_counter() - t0)

        avg = afw.get_average_field_array()
        s21 = afw.get_s21_array()

        clear_output(wait=True)
        print("batches_done:", afw.get_batches_done(), "/", afw.get_batches_total())
        print("averages_done:", afw.get_averages_done(), "/", afw.get_averages_total())
        print("last chunk elapsed:", f"{times[-1]:.3f} s")
        print("average shape/dtype:", avg.shape, avg.dtype)
        print("s21:", s21)

        fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
        for field_idx in range(avg.shape[0]):
            axes[0].plot(np.abs(avg[field_idx]), label=f"field {field_idx}")
        axes[0].set_title("|average field|")
        axes[0].legend(loc="best")

        if RESULT_MODE in ("average_g1", "all_correlators"):
            g1 = afw.get_g1_correlator_array()
            axes[1].plot(np.abs(np.diag(g1)))
            axes[1].set_title("|diag(g1)|")
        else:
            axes[1].plot(np.abs(avg[0]))
            axes[1].set_title("|average field 0|")

        plt.tight_layout()
        display(fig)
        plt.close(fig)
finally:
    if afw.is_fifo_active():
        afw.stop_fifo()

print("done")
print("mean chunk elapsed:", np.mean(times) if times else None)

In [ ]:
avg = afw.get_average_field_array()
s21 = afw.get_s21_array()
sub_data = afw.get_subtraction_data_array()
sub_trace = afw.get_subtraction_trace_array()

print("avg:", avg.shape, avg.dtype, np.isfinite(avg).all())
print("s21:", s21.shape, s21.dtype, np.isfinite(s21).all(), s21)
print("subtraction data:", sub_data.shape, sub_data.dtype, np.isfinite(sub_data).all())
print("subtraction trace:", sub_trace.shape, sub_trace.dtype, np.isfinite(sub_trace).all())

if RESULT_MODE in ("average_g1", "all_correlators"):
    g1 = afw.get_g1_correlator_array()
    print("g1:", g1.shape, g1.dtype, np.isfinite(g1).all())

if RESULT_MODE == "all_correlators":
    print("g1 other:", afw.get_g1_other_correlators_array().shape)
    print("cross_power:", afw.get_cross_power_array().shape)
    print("cross_spectrum:", afw.get_cross_spectrum_array().shape)

In [ ]:
# Run when done.
try:
    afw.free()
except Exception as exc:
    print("afw.free() failed:", exc)

try:
    dig.close()
except Exception as exc:
    print("dig.close() failed:", exc)